# 00 · Colab setup — DarkSight
Run top to bottom once per session. Runtime → Change runtime type → **T4 GPU**.

Layout on Google Drive (shared folder, create once):
```
MyDrive/darksight/
├── downloads/   ExDark.zip, ExDark_Annno.zip, (LOLdataset.zip)
├── data/        exdark_yolo_raw.zip, exdark_yolo_enhanced.zip   (prepared datasets, zipped)
├── runs/        Ultralytics training runs (checkpoints survive disconnects)
└── weights/     det_raw.pt, det_enh.pt
```
Datasets are **unzipped to /content** (local SSD) for speed; only runs/weights are written to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/darksight'
!mkdir -p {DRIVE}/downloads {DRIVE}/data {DRIVE}/runs {DRIVE}/weights
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!git clone https://github.com/<your-team>/darksight.git darksight || (cd darksight && git pull)
%cd /content/darksight
!pip -q install "ultralytics>=8.3,<9" scikit-image gdown gradio
import torch, ultralytics; print(torch.__version__, ultralytics.__version__, torch.cuda.is_available())

## Download ExDark (first time only — ~1.5 GB, cached on Drive)

In [ ]:
import os
if not os.path.exists(f'{DRIVE}/downloads/ExDark.zip'):
    !gdown 1BHmPgu8EsHoFDDkMGLVoXIlCth2dW6Yx -O {DRIVE}/downloads/ExDark.zip
    !gdown 1P3iO3UYn7KoBi5jiUkogJq96N6maZS1i -O {DRIVE}/downloads/ExDark_Annno.zip
!ls -lh {DRIVE}/downloads

## Prepare the YOLO dataset (first time only), then cache it on Drive as a zip

In [ ]:
import os
if not os.path.exists(f'{DRIVE}/data/exdark_yolo_raw.zip'):
    !mkdir -p /content/src && unzip -q -o {DRIVE}/downloads/ExDark.zip -d /content/src && unzip -q -o {DRIVE}/downloads/ExDark_Annno.zip -d /content/src
    !python scripts/prepare_exdark.py --images /content/src/ExDark --annotations /content/src/ExDark_Annno --out data/exdark_yolo/raw --workers 2
    !cd data/exdark_yolo && zip -q -r -0 {DRIVE}/data/exdark_yolo_raw.zip raw
    !cp data/exdark_meta.csv reports/data_stats.md {DRIVE}/data/
else:
    !mkdir -p data/exdark_yolo reports && unzip -q -o {DRIVE}/data/exdark_yolo_raw.zip -d data/exdark_yolo
    !cp {DRIVE}/data/exdark_meta.csv data/ && cp {DRIVE}/data/data_stats.md reports/
!python scripts/make_data_yaml.py data/exdark_yolo/raw
!ls data/exdark_yolo/raw/images/*/ | head; cat data/exdark_yolo/raw/data.yaml

In [ ]:
!python -m pytest -q -m 'not slow'